In [1]:
from pathlib import Path
from tqdm import tqdm
import SimpleITK as sitk
import numpy as np
import pandas as pd

BASE_ROOT = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed")
src_root = BASE_ROOT / "1initNifti"

CT_BACKGROUND = -1024
MR_BACKGROUND = 0
MASK_BACKGROUND = 0

In [2]:
def get_first_nifti(folder: Path):
    if not folder.exists():
        return None
    for pattern in ("*.nii.gz", "*.nii"):
        files = sorted(folder.glob(pattern))
        if files:
            return files[0]
    return None


In [3]:
def collect_metadata(src_root: Path) -> pd.DataFrame:
    rows = []

    for case_dir in tqdm(sorted(src_root.iterdir())):
        if not case_dir.is_dir():
            continue

        case_id = case_dir.name

        ct_in = get_first_nifti(case_dir / "CT_reg")
        mr_in = get_first_nifti(case_dir / "MR")
        mask_in = get_first_nifti(case_dir / "masks")

        def process_one(modality: str, in_path: Path):
            img = sitk.ReadImage(str(in_path))
            arr = sitk.GetArrayFromImage(img)  # (z, y, x)
            z, y, x = arr.shape

            sx, sy, sz = img.GetSpacing()  # (sx, sy, sz) in physical coords

            rows.append({
                "case_id": case_id,
                "modality": modality,
                "filename": in_path.name,
                "z": z,
                "y": y,
                "x": x,
                "spacing_x": sx,
                "spacing_y": sy,
                "spacing_z": sz,
            })

        if ct_in is not None:
            process_one("CT", ct_in)
        if mr_in is not None:
            process_one("MR", mr_in)
        if mask_in is not None:
            process_one("MASK", mask_in)

    return pd.DataFrame(rows)


In [4]:
df = collect_metadata(src_root)
df.head()


  0%|          | 0/938 [00:00<?, ?it/s]

100%|██████████| 938/938 [06:49<00:00,  2.29it/s]


,case_id,modality,filename,z,y,x,spacing_x,spacing_y,spacing_z
0,AB_1ABA005,CT,AB_1ABA005_CT_reg.nii.gz,91,367,465,1.0,1.0,3.0
1,AB_1ABA005,MR,AB_1ABA005_MR.nii.gz,91,367,465,1.0,1.0,3.0
2,AB_1ABA005,MASK,AB_1ABA005_mask.nii.gz,91,367,465,1.0,1.0,3.0
3,AB_1ABA009,CT,AB_1ABA009_CT_reg.nii.gz,110,389,494,1.0,1.0,3.0
4,AB_1ABA009,MR,AB_1ABA009_MR.nii.gz,110,389,494,1.0,1.0,3.0


In [5]:
df.to_csv(BASE_ROOT / "dataset_shape_spacing_stats.csv", index=False)

In [6]:
df = pd.read_csv(BASE_ROOT / "dataset_shape_spacing_stats.csv")
df.head()

df.sample(20)

,case_id,modality,filename,z,y,x,spacing_x,spacing_y,spacing_z
1800,brain_1BA141,CT,brain_1BA141_CT_reg.nii.gz,190,257,183,1.0,1.0,1.0
1,AB_1ABA005,MR,AB_1ABA005_MR.nii.gz,91,367,465,1.0,1.0,3.0
1458,TH_1THA293,CT,TH_1THA293_CT_reg.nii.gz,80,474,465,1.0,1.0,3.0
1251,TH_1THA041,CT,TH_1THA041_CT_reg.nii.gz,105,417,559,1.0,1.0,3.0
550,HN_1HNA015,MR,HN_1HNA015_MR.nii.gz,115,379,499,1.0,1.0,3.0
1384,TH_1THA261,MR,TH_1THA261_MR.nii.gz,89,479,467,1.0,1.0,3.0
2418,pelvis_1PA084,CT,pelvis_1PA084_CT_reg.nii.gz,120,299,447,1.0,1.0,2.5
172,AB_1ABA111,MR,AB_1ABA111_MR.nii.gz,104,345,446,1.0,1.0,3.0
994,HN_1HND001,MR,HN_1HND001_MR.nii.gz,88,280,273,1.0,1.0,3.0
2127,brain_1BC021,CT,brain_1BC021_CT_reg.nii.gz,167,262,224,1.0,1.0,1.0


In [9]:
df = df[df["modality"]=="MR"]
df["body_part"] = df["case_id"].str.split("_").str[0]

desc = df.groupby("body_part")[["x", "y", "z"]].describe()

In [28]:
df.groupby("body_part")[["spacing_z", "spacing_y", "spacing_x"]].describe()

spacing_z                                    spacing_y       ...  \
              count mean  std  min  25%  50%  75%  max     count mean  ...   
body_part                                                              ...   
AB             21.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0      21.0  1.0  ...   
HN             39.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0      39.0  1.0  ...   
TH             24.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0      24.0  1.0  ...   
brain         540.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0     540.0  1.0  ...   
pelvis        540.0  2.5  0.0  2.5  2.5  2.5  2.5  2.5     540.0  1.0  ...   

                    spacing_x                                     
           75%  max     count mean  std  min  25%  50%  75%  max  
body_part                                                         
AB         1.0  1.0      21.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
HN         1.0  1.0      39.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
TH         1.0  1.0      24.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
brain      1.0  1.0     540.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
pelvis     1.0  1.0     540.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  

[5 rows x 24 columns]

In [ ]:
def extract_hosp_char(case_id: str, body_part: str) -> str:
    s = case_id.upper()
    
    # mapping from body_part to the prefix we search in the case_id
    prefix_map = {
        "pelvis": "P",
        "brain": "B",
        "TH": "TH",
        "HN": "HN",
        "AB": "AB",
    }

    prefix = prefix_map.get(body_part, body_part)  # fallback: use body_part itself
    idx = s.rfind(prefix)

    pos = idx + len(prefix)
    if pos < len(s):
        hospital = s[pos]
        if body_part in ("pelvis", "brain"):
            return "2023" + hospital
        else:
            return "2025" + hospital
    else:
        return ""

df["hosp_char"] = df.apply(
    lambda row: extract_hosp_char(row["case_id"], row["body_part"]),
    axis=1,
)

In [25]:
df.groupby(["hosp_char", "body_part"])["spacing_x"].describe()

count  mean  std  min  25%  50%  75%  max
hosp_char body_part                                           
20230     brain      126.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
20231     brain       48.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
20232     brain        6.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2023A     brain      180.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          pelvis     360.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2023C     brain      180.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          pelvis     180.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2025A     TH          24.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2025C     AB          21.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          HN          39.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0